In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-5.6-luna",  # 대안: "gpt-4o" "gpt-4o-mini"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: ChatOpenAI 에 그대로 전달할 추가 옵션 (예: timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

#### 1. CommaSeparatedListOutputParser

In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import csv

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [4]:
# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# LLM 설정 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0.7)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

# "AI 관련 기술"에 대한 체인 호출 실행
result = chain.invoke({"subject": "AI 관련 기술"})

# 쉼표로 구분된 리스트 출력
print("AI 관련 기술 목록:")
print(result)

AI 관련 기술 목록:
['머신러닝', '딥러닝', '자연어 처리', '컴퓨터 비전', '강화학습']


In [6]:
# 결과 활용 예시: CSV 파일로 저장
csv_filename = "../data/ai_technologies.csv"
with open(csv_filename, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["AI 기술"])  # 헤더 추가
    for item in result:
        writer.writerow([item])

print(f" '{csv_filename}' 파일로 저장 완료!")


 '../data/ai_technologies.csv' 파일로 저장 완료!


#### 2. JsonOutputParser

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import json

# JSON 출력 파서 초기화
parser = JsonOutputParser()

# 프롬프트 템플릿을 설정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 과학 분야 전문가 AI입니다. 질문에 대해 체계적이고 간결한 답변을 JSON 형식으로 제공하세요. "
                   "JSON의 키 이름은 질문에서 지정한 영문 키를 그대로 사용하고, 값은 반드시 한국어로 작성하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# JSON 출력 형식 지침을 프롬프트에 적용
prompt = prompt.partial(format_instructions=parser.get_format_instructions())


# 프롬프트, 모델, 출력 파서를 연결하는 체인 생성
chain = prompt | llm | parser

# 질문 설정 (우주 탐사 관련 질문)
question = "최근 10년간 진행된 주요 우주 탐사 미션 3가지를 알려주세요. \
각 미션의 이름은 `mission_name`에, 목표는 `goal`에, 주관 기관은 `agency`에 담아 주세요."

# 체인 실행 및 JSON 응답 받기
response = chain.invoke({"question": question})
print(type(response))

# JSON 데이터 출력
print(json.dumps(response, indent=4, ensure_ascii=False))


<class 'dict'>
{
    "missions": [
        {
            "mission_name": "퍼서비어런스 로버 (Mars 2020)",
            "goal": "화성 표면 탐사 및 과거 미생물 존재 가능성 조사",
            "agency": "미국 항공우주국(NASA)"
        },
        {
            "mission_name": "솔라 오비터",
            "goal": "태양의 자기장과 플라즈마 흐름을 관측하여 태양 활동 메커니즘 규명",
            "agency": "유럽우주국(ESA) 및 미국 항공우주국(NASA) 공동"
        },
        {
            "mission_name": "천문위성 톈원-1",
            "goal": "화성 궤도, 착륙, 로버를 통한 종합 화성 탐사와 지질·대기 연구",
            "agency": "중국 국가우주국(CNSA)"
        }
    ]
}


#### 3. PandasDataFrameOutputParser
* langchain 1.2 버전에서는 PandasDataFrameOutputParser 가 삭제 되었음

In [8]:
# import pandas as pd
# from langchain_core.output_parsers import PandasDataFrameOutputParser
# from langchain_core.prompts import PromptTemplate
# from langchain_openai import ChatOpenAI
# import re

# # Titanic 데이터셋 로드
# df = pd.read_csv('../data/titanic.csv')

# # ChatOpenAI 모델 초기화
# #llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")

# # Pandas DataFrame Output Parser 설정
# parser = PandasDataFrameOutputParser(dataframe=df)

# # 형식 지침 출력
# format_instructions = parser.get_format_instructions()
# print("Format Instructions:\n", format_instructions)

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")

# # 프롬프트 템플릿 설정
# prompt = PromptTemplate(
#     template=""" 
#     You are a helpful assistant that interacts with a Pandas DataFrame.
#     The DataFrame contains the following columns: {columns}.
    
#     Your task is to answer the user's query by generating a command in the following format:
#     {format_instructions}
    
#     User Query: {query}    
#     """,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": format_instructions,
#         "columns": ", ".join(df.columns)
#     },
# )

# # 체인 생성
# chain = prompt | llm | parser

# # 모델 응답 받기
# try:
#     # **Name 열을 표시하십시오.**
#     print('Name 컬럼 출력')
#     df_query = "Show the Name column"

#     parser_output = chain.invoke({"query": df_query})
#     print(type(parser_output))
#     print(parser_output)

#     # **첫번째 행을 표시하십시오.**
#     print('첫번째 행 출력')
#     df_query2 = "Show first row"

#     parser_output2 = chain.invoke({"query": df_query2})
#     print(parser_output2)

#     #Please tell me the average value of the Fare column.
#     print('Fare 컬럼의 평균값 출력')
#     df_query3 = "Show me the average value of the Fare column."

#     parser_output3 = chain.invoke({"query": df_query3})
#     print(parser_output3)

# except Exception as e:
#     print(f"오류 발생: {e}")


#### PydanticOutputParser를 사용하여 다시 수정한 예제

In [9]:
import pandas as pd
import re
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Titanic 데이터셋 로드
df = pd.read_csv('../data/titanic.csv')

# Pydantic 모델 정의 (pandas 명령어 구조화)
class PandasCommand(BaseModel):
    """Pandas DataFrame에서 실행할 명령어"""
    command: str = Field(..., description="실행할 pandas 명령어 예: df['Name'], df.iloc[0], df['Fare'].mean()")

# LLM 초기화 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0.7, provider="openai")

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=PandasCommand)

# 형식 지침 출력
format_instructions = parser.get_format_instructions()
print("Format Instructions:\n", format_instructions)
print("-" * 50)

# 프롬프트 템플릿 설정 (ChatPromptTemplate 사용 권장)
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a pandas DataFrame expert for the Titanic dataset.
    
Columns: {columns}
    
User query에 맞는 **pandas 명령어만** 생성하세요. 다른 설명 없음."""),
    ("user", "{query}\n\n{format_instructions}")
])

# 입력 변수 부분 설정
prompt = prompt.partial(
    columns=", ".join(df.columns),
    format_instructions=format_instructions
)

# 체인 생성
chain = prompt | llm | parser

# 안전한 pandas 명령 실행 함수
def safe_execute_pandas(df, command: str):
    """허용된 pandas 명령어만 안전하게 실행"""
    safe_commands = {
        r"df\['[^']*'\]": lambda: eval(command, {"df": df}, {}),
        r'df\["[^"]*"\]': lambda: eval(command, {"df": df}, {}),
        r"df\.iloc\[\d+\]": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.mean\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\['[^']*'\]\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.head\(\)": lambda: eval(command, {"df": df}, {}),
        r"df\.shape": lambda: eval(command, {"df": df}, {}),
    }
    
    command = command.strip()
    for pattern, executor in safe_commands.items():
        if re.match(pattern, command):
            try:
                return executor()
            except Exception as e:
                return f"실행 오류: {e}"
    return "허용되지 않은 명령어입니다."

# 테스트 실행
queries = [
    ("Name 컬럼 출력", "Show the Name column"),
    ("첫번째 행 출력", "Show first row"),
    ("Fare 컬럼 평균값 출력", "Show me the average value of the Fare column.")
]

for title, query in queries:
    print(f"\n{title}")
    print("=" * 40)
    
    try:
        result = chain.invoke({"query": query})
        print("생성된 명령어:", result.command)
        print("실행 결과:")
        output = safe_execute_pandas(df, result.command)
        print(output)
        print(type(output))
        
    except Exception as e:
        print(f"오류 발생: {e}")
        print(type(e))



[LLM 생성] provider=openai | model=gpt-5.6-luna
Format Instructions:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Pandas DataFrame에서 실행할 명령어", "properties": {"command": {"description": "실행할 pandas 명령어 예: df['Name'], df.iloc[0], df['Fare'].mean()", "title": "Command", "type": "string"}}, "required": ["command"]}
```
--------------------------------------------------

Name 컬럼 출력
생성된 명령어: df['Name']
실행 결과:
0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina


In [10]:
import pandas as pd
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# LLM 초기화 (제공자/키/모델명은 공통 설정 셀의 get_llm 사용)
llm = get_llm(temperature=0, seed=42, timeout=60, provider="openai")  # seed 고정으로 재현성 향상, timeout 60초


[LLM 생성] provider=openai | model=gpt-5.6-luna


In [11]:

# Pydantic 모델 정의 (테이블 데이터 구조화)
class TableData(BaseModel):
    """테이블 행을 나타내는 딕셔너리 리스트"""
    data: list[dict] = Field(..., description="테이블 행 데이터 (딕셔너리 리스트)") # [{}, {}, ...]

# PydanticOutputParser 설정
parser = PydanticOutputParser(pydantic_object=TableData)

# ChatPromptTemplate 설정 (최신 LCEL 스타일)
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 테이블 형태의 데이터를 생성하는 AI 어시스턴트입니다.
    매번 **동일한 데이터**를 정확히 동일한 순서로 반환하세요.
    반드시 다음 JSON 스키마를 정확히 따라야 합니다:
    {format_instructions}
    
    데이터는 현실적이고 일관성 있게 생성하세요. (실제 통계가 아닌 예시용 가상 데이터입니다.)"""),
    ("user", "{query}")
])

# 부분 변수 설정
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 체인 생성
chain = prompt | llm | parser

# DataFrame 생성 함수
def generate_dataframe(user_query: str, expected_columns: list[str] | None = None) -> pd.DataFrame:
    """LLM으로 테이블 데이터를 생성해 DataFrame으로 반환합니다. 실패 시 빈 DataFrame을 반환합니다."""
    print(" 쿼리 처리 중:", user_query[:50] + "...")
    try:
        # 모델 호출
        result = chain.invoke({"query": user_query})
        print(" 생성된 JSON 데이터:", result)

        # DataFrame으로 변환
        df = pd.DataFrame(result.data)

        # 요청한 컬럼과 다르면 실패 처리
        if expected_columns and set(df.columns) != set(expected_columns):
            print(f" 컬럼 불일치: 기대 {expected_columns} / 실제 {list(df.columns)}")
            return pd.DataFrame()

        print("\n 생성된 DataFrame:")
        print(df)
        print(f"\n 데이터 형태: {df.shape} ({len(df.columns)} columns)")
        return df

    except Exception as e:
        print(f" 오류 발생: {type(e).__name__}: {e}")
        return pd.DataFrame()


In [12]:

# 실행 테스트
print("2024년 상반기 서울 아파트 평균 매매 가격 데이터 생성")
print("=" * 60)

df_seoul_housing = generate_dataframe(
    "2024년 상반기 서울 아파트 평균 매매가격 데이터셋을 생성해주세요. "
    "JSON 키는 반드시 영어로 정확히 다음 이름을 사용하세요: "
    "District, Average_Price_100M_KRW, Transactions, YoY_Change_Percent. "
    "District 값은 서울 25개 구의 한글 이름으로 작성하세요.",
    expected_columns=["District", "Average_Price_100M_KRW", "Transactions", "YoY_Change_Percent"],
)

print("\n" + "=" * 60)
print("최종 결과:")
print(df_seoul_housing)

2024년 상반기 서울 아파트 평균 매매 가격 데이터 생성
 쿼리 처리 중: 2024년 상반기 서울 아파트 평균 매매가격 데이터셋을 생성해주세요. JSON 키는 반드시...
 생성된 JSON 데이터: data=[{'District': '강남구', 'Average_Price_100M_KRW': 28.4, 'Transactions': 1248, 'YoY_Change_Percent': 3.8}, {'District': '강동구', 'Average_Price_100M_KRW': 13.2, 'Transactions': 982, 'YoY_Change_Percent': 2.6}, {'District': '강북구', 'Average_Price_100M_KRW': 7.1, 'Transactions': 438, 'YoY_Change_Percent': 1.4}, {'District': '강서구', 'Average_Price_100M_KRW': 8.9, 'Transactions': 1056, 'YoY_Change_Percent': 1.9}, {'District': '관악구', 'Average_Price_100M_KRW': 8.4, 'Transactions': 721, 'YoY_Change_Percent': 2.1}, {'District': '광진구', 'Average_Price_100M_KRW': 14.8, 'Transactions': 634, 'YoY_Change_Percent': 3.2}, {'District': '구로구', 'Average_Price_100M_KRW': 8.2, 'Transactions': 847, 'YoY_Change_Percent': 1.7}, {'District': '금천구', 'Average_Price_100M_KRW': 7.3, 'Transactions': 496, 'YoY_Change_Percent': 1.2}, {'District': '노원구', 'Average_Price_100M_KRW': 7.4, 'Transactions': 1135, 'YoY

In [13]:
df_seoul_housing

,District,Average_Price_100M_KRW,Transactions,YoY_Change_Percent
0,강남구,28.4,1248,3.8
1,강동구,13.2,982,2.6
2,강북구,7.1,438,1.4
3,강서구,8.9,1056,1.9
4,관악구,8.4,721,2.1
5,광진구,14.8,634,3.2
6,구로구,8.2,847,1.7
7,금천구,7.3,496,1.2
8,노원구,7.4,1135,1.5
9,도봉구,6.8,612,0.9


In [14]:
print('2024년 서울 지하철역별 유동 인구 데이터')
# [예제 2] 2024년 서울 지하철역별 유동 인구 데이터
df_seoul_subway = generate_dataframe(
    #"Generate a dataset of the top 10 busiest subway stations in Seoul in 2024 with columns: Station Name, Line Number, Daily Passenger Volume, and Weekday vs Weekend Ratio."
    "2024년 서울 지하철 승객 이용량이 가장 많은 상위 10개 역의 데이터셋을 생성해주세요. \
        컬럼은 다음과 같습니다: 역명(Station_Name), 호선(Line_Number),\
              일평균승객수_만명(Daily_Passengers_10K), 평일대주말비율(Weekday_Weekend_Ratio).\
                  실제 서울 지하철의 주요 역들을 기반으로 현실적인 데이터를 생성해주세요.",
    expected_columns=["Station_Name", "Line_Number", "Daily_Passengers_10K", "Weekday_Weekend_Ratio"],
)

#if df_seoul_subway is not None:
df_seoul_subway.head()


2024년 서울 지하철역별 유동 인구 데이터
 쿼리 처리 중: 2024년 서울 지하철 승객 이용량이 가장 많은 상위 10개 역의 데이터셋을 생성해주세요....
 생성된 JSON 데이터: data=[{'Station_Name': '홍대입구', 'Line_Number': '2호선·경의중앙선·공항철도', 'Daily_Passengers_10K': 10.8, 'Weekday_Weekend_Ratio': 0.96}, {'Station_Name': '강남', 'Line_Number': '2호선·신분당선', 'Daily_Passengers_10K': 10.4, 'Weekday_Weekend_Ratio': 1.42}, {'Station_Name': '잠실', 'Line_Number': '2호선·8호선', 'Daily_Passengers_10K': 9.7, 'Weekday_Weekend_Ratio': 1.18}, {'Station_Name': '서울역', 'Line_Number': '1호선·4호선·경의중앙선·공항철도', 'Daily_Passengers_10K': 9.1, 'Weekday_Weekend_Ratio': 1.29}, {'Station_Name': '고속터미널', 'Line_Number': '3호선·7호선·9호선', 'Daily_Passengers_10K': 8.3, 'Weekday_Weekend_Ratio': 1.21}, {'Station_Name': '사당', 'Line_Number': '2호선·4호선', 'Daily_Passengers_10K': 7.2, 'Weekday_Weekend_Ratio': 1.34}, {'Station_Name': '건대입구', 'Line_Number': '2호선·7호선', 'Daily_Passengers_10K': 7.0, 'Weekday_Weekend_Ratio': 1.12}, {'Station_Name': '신림', 'Line_Number': '2호선·신림선', 'Daily_Passengers_10K': 6.8, 'Weekday_

,Station_Name,Line_Number,Daily_Passengers_10K,Weekday_Weekend_Ratio
0,홍대입구,2호선·경의중앙선·공항철도,10.8,0.96
1,강남,2호선·신분당선,10.4,1.42
2,잠실,2호선·8호선,9.7,1.18
3,서울역,1호선·4호선·경의중앙선·공항철도,9.1,1.29
4,고속터미널,3호선·7호선·9호선,8.3,1.21


In [15]:
df_seoul_subway

,Station_Name,Line_Number,Daily_Passengers_10K,Weekday_Weekend_Ratio
0,홍대입구,2호선·경의중앙선·공항철도,10.8,0.96
1,강남,2호선·신분당선,10.4,1.42
2,잠실,2호선·8호선,9.7,1.18
3,서울역,1호선·4호선·경의중앙선·공항철도,9.1,1.29
4,고속터미널,3호선·7호선·9호선,8.3,1.21
5,사당,2호선·4호선,7.2,1.34
6,건대입구,2호선·7호선,7.0,1.12
7,신림,2호선·신림선,6.8,1.37
8,구로디지털단지,2호선,6.6,1.56
9,삼성,2호선,6.3,1.61


In [16]:
print('한국 5대 편의점 브랜드별 2024년 매출 및 점포 수')
# [예제 3] 한국 5대 편의점 브랜드별 2024년 매출 및 점포 수
df_korean_convenience_stores = generate_dataframe(
    #"Create a dataset of the top 5 convenience store brands in Korea in 2024 with columns: Brand Name, Number of Stores, Total Revenue (in billion KRW), and Market Share (%)."
    "2024년 한국의 편의점 브랜드 상위 5개사 데이터셋을 생성해주세요.\
          컬럼은 다음과 같습니다: 브랜드명(Brand_Name), 점포수(Store_Count), \
            총매출_조원(Revenue_Trillion_KRW), 시장점유율_퍼센트(Market_Share_Percent).\
                  CU, GS25, 세븐일레븐, 이마트24, 미니스톱 등 실제 한국 편의점 브랜드를 기반으로 현실적인 데이터를 생성해주세요.",
    expected_columns=["Brand_Name", "Store_Count", "Revenue_Trillion_KRW", "Market_Share_Percent"],
)


한국 5대 편의점 브랜드별 2024년 매출 및 점포 수
 쿼리 처리 중: 2024년 한국의 편의점 브랜드 상위 5개사 데이터셋을 생성해주세요.          컬럼...
 생성된 JSON 데이터: data=[{'Brand_Name': 'CU', 'Store_Count': 18458, 'Revenue_Trillion_KRW': 8.75, 'Market_Share_Percent': 34.0}, {'Brand_Name': 'GS25', 'Store_Count': 18112, 'Revenue_Trillion_KRW': 8.62, 'Market_Share_Percent': 33.5}, {'Brand_Name': '세븐일레븐', 'Store_Count': 12500, 'Revenue_Trillion_KRW': 4.45, 'Market_Share_Percent': 17.3}, {'Brand_Name': '이마트24', 'Store_Count': 6579, 'Revenue_Trillion_KRW': 2.58, 'Market_Share_Percent': 10.0}, {'Brand_Name': '미니스톱', 'Store_Count': 2600, 'Revenue_Trillion_KRW': 1.34, 'Market_Share_Percent': 5.2}]

 생성된 DataFrame:
  Brand_Name  Store_Count  Revenue_Trillion_KRW  Market_Share_Percent
0         CU        18458                  8.75                  34.0
1       GS25        18112                  8.62                  33.5
2      세븐일레븐        12500                  4.45                  17.3
3      이마트24         6579                  2.58             

In [17]:

df_korean_convenience_stores

,Brand_Name,Store_Count,Revenue_Trillion_KRW,Market_Share_Percent
0,CU,18458,8.75,34.0
1,GS25,18112,8.62,33.5
2,세븐일레븐,12500,4.45,17.3
3,이마트24,6579,2.58,10.0
4,미니스톱,2600,1.34,5.2


#### 4. EnumOutputParser

- 모델의 답변을 **미리 정한 값(Enum) 중 하나**로 제한할 때 사용합니다. (분류, 감성 분석, 카테고리 지정 등)
- langchain 1.x에서는 `langchain_core`가 아니라 **`langchain_classic.output_parsers`** 에서 import 합니다.
- 응답이 Enum 값에 없으면 `OutputParserException`이 발생합니다.
- 여러 필드가 있는 구조화된 결과가 필요하면 `PydanticOutputParser`를 사용합니다.

In [18]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate


# 허용할 분류 값을 Enum 으로 정의 (이름 = 코드에서 쓰는 값, 값 = 모델이 답해야 하는 문자열)
class Sentiment(Enum):
    """리뷰 감성 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "중립"


# Enum 출력 파서 초기화
parser = EnumOutputParser(enum=Sentiment)

# 출력 형식 지침 확인: "Select one of the following options: 긍정, 부정, 중립"
format_instructions = parser.get_format_instructions()
print(format_instructions)

# 모델 없이 파서만 테스트 (앞뒤 공백/줄바꿈은 제거하고 Enum 으로 변환)
print(repr(parser.parse("긍정")))
print(repr(parser.parse(" 부정\n")))

# 프롬프트 템플릿 설정 (형식 지침을 미리 주입)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 리뷰 감성 분류 전문가입니다. 리뷰를 읽고 감성을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

# LLM 초기화 (분류 작업이므로 temperature 를 0으로 설정)
llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → Enum 파서
chain = prompt | llm | parser

Select one of the following options: 긍정, 부정, 중립
<Sentiment.POSITIVE: '긍정'>
<Sentiment.NEGATIVE: '부정'>
[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [19]:
from langchain_core.exceptions import OutputParserException

reviews = [
    "배송도 빠르고 품질도 기대 이상이에요. 정말 만족합니다!",
    "사용한 지 이틀 만에 고장 났어요. 환불하고 싶습니다.",
    "그냥 평범한 제품이에요. 가격만큼 하는 것 같아요.",
]

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = chain.invoke({"review": review})
        # 결과는 문자열이 아니라 Sentiment Enum 멤버
        print(f"  분류 결과: {result} / 타입: {type(result).__name__} / 값: {result.value}")
    except OutputParserException as e:
        # 모델이 Enum 에 없는 값으로 답하면 파싱 오류가 발생
        print(f"  파싱 오류: {e}")
    print("-" * 50)

리뷰: 배송도 빠르고 품질도 기대 이상이에요. 정말 만족합니다!
  분류 결과: Sentiment.POSITIVE / 타입: Sentiment / 값: 긍정
--------------------------------------------------
리뷰: 사용한 지 이틀 만에 고장 났어요. 환불하고 싶습니다.
  분류 결과: Sentiment.NEGATIVE / 타입: Sentiment / 값: 부정
--------------------------------------------------
리뷰: 그냥 평범한 제품이에요. 가격만큼 하는 것 같아요.
  분류 결과: Sentiment.NEUTRAL / 타입: Sentiment / 값: 중립
--------------------------------------------------


##### EnumOutputParser + OutputFixingParser (파싱 오류 자동 복구)

- `OutputFixingParser`는 기존 파서를 감싸서, 파싱에 실패(`OutputParserException`)하면 **잘못된 응답 + 오류 내용 + 형식 지침을 LLM에 다시 보내 수정된 응답을 받아** 재파싱합니다.
- `langchain_classic.output_parsers`에서 import 하며, `max_retries`(기본 1)만큼 수정을 시도합니다.
- **주의**: 파싱이 실패할 때만 **LLM이 추가로 호출**되므로 비용과 응답 시간이 늘어납니다. 수정용 LLM은 `from_llm(llm=...)`에서 별도로 지정할 수 있습니다. (예: 더 저렴한 모델)
- 수정 후에도 형식에 맞지 않으면 `OutputParserException`이 다시 발생하므로 `try/except`는 여전히 필요합니다.

In [ ]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

# 기존 Enum 파서를 감싸서 파싱 실패 시 LLM 으로 응답을 자동 수정 (max_retries: 수정 시도 횟수)
fixing_parser = OutputFixingParser.from_llm(llm=llm, parser=parser, max_retries=1)

# 분류 정보가 전혀 없는 응답 (모델이 지침을 어기고 값을 정하지 못한 경우를 가정)
# 기본 파서는 오류를 내고, OutputFixingParser 는 원래 리뷰를 알 수 없어 값을 추측해서 채움
bad_completion = "Null Value"

# 1) 기본 Enum 파서만 사용: 오류 발생
try:
    parser.parse(bad_completion)
except OutputParserException as e:
    print("1) 기본 파서 오류:", str(e).splitlines()[0])

# 2) OutputFixingParser 사용: 오류를 LLM 에게 알려 수정된 응답으로 재파싱 (LLM 호출 1회 발생)
# 주의: 응답에 근거가 없으므로 아래 결과는 추측이며 실행할 때마다 달라질 수 있음
#      (응답에 의미가 있는 "Positive" 같은 경우는 "긍정"으로 안정적으로 복구됨)
fixed = fixing_parser.parse(bad_completion)
print(f"2) 수정 후 결과: {fixed} / 값: {fixed.value}")

기본 파서 오류: Response 'Null Value' is not one of the expected values: ['긍정', '부정', '중립']
수정 후 결과: Sentiment.NEUTRAL / 값: 중립


In [21]:
# 체인에 OutputFixingParser 연결: 프롬프트 → 모델 → 오류 자동 복구 파서
fixing_chain = prompt | llm | fixing_parser

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = fixing_chain.invoke({"review": review})
        print(f"  분류 결과: {result} / 값: {result.value}")
    except OutputParserException as e:
        # 수정 시도 후에도 Enum 값이 아니면 오류 발생
        print(f"  파싱 오류(수정 실패): {str(e).splitlines()[0]}")
    print("-" * 50)

리뷰: 배송도 빠르고 품질도 기대 이상이에요. 정말 만족합니다!
  분류 결과: Sentiment.POSITIVE / 값: 긍정
--------------------------------------------------
리뷰: 사용한 지 이틀 만에 고장 났어요. 환불하고 싶습니다.
  분류 결과: Sentiment.NEGATIVE / 값: 부정
--------------------------------------------------
리뷰: 그냥 평범한 제품이에요. 가격만큼 하는 것 같아요.
  분류 결과: Sentiment.NEUTRAL / 값: 중립
--------------------------------------------------


#### 5. StructuredOutputParser + ResponseSchema

- 출력할 필드를 `ResponseSchema(name, description)`로 정의하면, 모델이 **JSON 형식으로 답하도록 지침을 만들고 응답을 `dict`로 변환**해 줍니다.
- langchain 1.x에서는 `langchain_core`가 아니라 **`langchain_classic.output_parsers`** 에서 import 합니다. (`StructuredOutputParser`, `ResponseSchema` 모두 해당)
- `ResponseSchema`의 `type`(기본값 `"string"`)은 **형식 지침에만 반영**될 뿐 값의 타입을 검증하지는 않습니다.
- 응답에 정의한 필드(키)가 빠져 있으면 `OutputParserException`이 발생합니다.
- 타입 검증(숫자 범위, 리스트 등)이 필요하면 `PydanticOutputParser`를 사용합니다.

In [22]:
import json
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate

# 출력할 필드를 ResponseSchema 로 정의 (name: 결과 dict 의 키, description: 모델에게 알려줄 설명)
response_schemas = [
    ResponseSchema(name="title", description="추천 영화 또는 드라마 제목"),
    ResponseSchema(name="genre", description="장르"),
    ResponseSchema(name="reason", description="추천 이유 (한 문장)"),
]

# ResponseSchema 목록으로 파서 생성
parser = StructuredOutputParser.from_response_schemas(response_schemas)

# 출력 형식 지침 확인: ```json 코드 블록과 필드별 설명이 포함됨
format_instructions = parser.get_format_instructions()
print(format_instructions)

# 모델 없이 파서만 테스트 (```json 코드 블록 응답을 dict 로 변환)
sample = '```json\n{"title": "인사이드 아웃", "genre": "애니메이션", "reason": "온 가족이 함께 공감하며 볼 수 있습니다."}\n```'
print(parser.parse(sample))

# 프롬프트 템플릿 설정 (형식 지침을 미리 주입)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화/드라마 추천 전문가입니다. 질문에 답하고 반드시 아래 형식을 지키세요.\n{format_instructions}"),
    ("user", "{question}"),
]).partial(format_instructions=format_instructions)

# LLM 초기화 (구조화 출력은 일관성이 중요하므로 temperature 를 0으로 설정)
llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → 구조화 파서
chain = prompt | llm | parser

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"title": string  // 추천 영화 또는 드라마 제목
	"genre": string  // 장르
	"reason": string  // 추천 이유 (한 문장)
}
```
{'title': '인사이드 아웃', 'genre': '애니메이션', 'reason': '온 가족이 함께 공감하며 볼 수 있습니다.'}
[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [23]:
from langchain_core.exceptions import OutputParserException

try:
    response = chain.invoke({"question": "주말에 가족과 함께 보기 좋은 영화나 드라마를 추천해 주세요."})
    print(type(response))  # 결과는 dict

    # JSON 데이터 출력
    print(json.dumps(response, indent=4, ensure_ascii=False))

    # 필드별 접근: ResponseSchema 의 name 이 dict 의 키
    print("추천 작품:", response["title"])
    print("장르:", response["genre"])
    print("추천 이유:", response["reason"])
except OutputParserException as e:
    # 모델이 형식을 지키지 않았거나 필드가 빠진 경우
    print(f"파싱 오류: {str(e).splitlines()[0]}")

<class 'dict'>
{
    "title": "인사이드 아웃 (Inside Out)",
    "genre": "애니메이션·가족·드라마",
    "reason": "감정을 의인화해 아이부터 어른까지 모두가 공감할 수 있는 따뜻한 이야기를 통해 가족 간의 소통과 이해를 돕는 영화라 주말에 함께 보기 좋습니다."
}
추천 작품: 인사이드 아웃 (Inside Out)
장르: 애니메이션·가족·드라마
추천 이유: 감정을 의인화해 아이부터 어른까지 모두가 공감할 수 있는 따뜻한 이야기를 통해 가족 간의 소통과 이해를 돕는 영화라 주말에 함께 보기 좋습니다.


In [24]:
# 상황이 다른 질문 3개로 상황에 맞는 추천을 하는지 확인 (같은 chain 재사용)
situations = [
    "주말에 가족과 함께 보기 좋은 영화나 드라마를 추천해 주세요.",
    "퇴근 후 혼자 가볍게 웃으며 볼 수 있는 코미디 드라마를 추천해 주세요.",
    "비 오는 날 밤에 보기 좋은 긴장감 있는 스릴러 영화를 추천해 주세요.",
]

for question in situations:
    print(f"질문: {question}")
    try:
        response = chain.invoke({"question": question})
        # 결과는 dict: ResponseSchema 의 name 이 키 (장르가 상황에 맞는지 확인)
        print(f"  추천 작품: {response['title']} / 장르: {response['genre']}")
        print(f"  추천 이유: {response['reason']}")
    except OutputParserException as e:
        # 모델이 형식을 지키지 않았거나 필드가 빠진 경우
        print(f"  파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)

질문: 주말에 가족과 함께 보기 좋은 영화나 드라마를 추천해 주세요.
  추천 작품: 인사이드 아웃 (Inside Out) / 장르: 애니메이션·가족·드라마
  추천 이유: 감정을 의인화해 아이부터 어른까지 모두가 공감할 수 있는 따뜻한 이야기를 통해 가족 간의 소통과 이해를 돕는 영화라 주말에 함께 보기 좋습니다.
--------------------------------------------------
질문: 퇴근 후 혼자 가볍게 웃으며 볼 수 있는 코미디 드라마를 추천해 주세요.
  추천 작품: 슬기로운 감빵생활 / 장르: 코미디·드라마
  추천 이유: 감옥이라는 독특한 배경 속에서 일어나는 유쾌한 인간관계와 위트 있는 대사들이 퇴근 후 가볍게 웃으며 즐기기 딱 좋은 작품입니다.
--------------------------------------------------
질문: 비 오는 날 밤에 보기 좋은 긴장감 있는 스릴러 영화를 추천해 주세요.
  추천 작품: 프리즌 (Prison) / 장르: 스릴러 / 범죄
  추천 이유: 비 오는 밤의 어두운 분위기와 맞물려, 감시 카메라와 교도소 내부의 숨막히는 긴장감이 지속적으로 고조돼 손에 땀을 쥐게 하는 영화입니다.
--------------------------------------------------
